In [10]:
from pydoc import describe

import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os
import copy





#132.4	133.8	133.
# 
# 
# 
# 	133.0	134.2	134.6




In [11]:
run_query = False

In [12]:
def run_sql(filename, sub_list=[], connection=None, filename_is_query=False):
    """
    Run a SQL Query by reading from a .txt file, substituting values when required.
    Input:
    filename (str): File that we want to read. Usually a .txt file.
    sub_list (list of (str,str) tuples): Substitute each instance of the first element of the tuple for the second.
                                         Example: [('{max_mob}', '6')]
    """
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for sub in sub_list:
        text, var = sub
        query = query.replace(text, var)
#     print(query)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df

In [13]:
if run_query == True:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
                # with open("establish_temp_tables_query.txt", "r") as file:
                #     temp_table_queries = file.read()
                # # conn.execute(temp_table_queries.strip())
                # print('temptables query finished')
        df = run_sql('weekly_query', connection=conn)
        print('loss query finished')
        bl = run_sql('b2l_query', connection=conn)
        print('book to look query finished')
        tminusone = run_sql('tminusone_query', connection=conn)
        ste_df = run_sql('ste_query', connection=conn)
        print('STE query finished')
        ste_bl = run_sql('ste_b2l', connection=conn)
        print('STE b2l query finished')
        
        # Save to cache for faster reloads
        df.to_pickle('df_cache.pkl')
        bl.to_pickle('bl_cache.pkl')
        tminusone.to_pickle('tminusone_cache.pkl')
        ste_df.to_pickle('ste_df_cache.pkl')
        ste_bl.to_pickle('ste_bl_cache.pkl')
        print('Data cached to pickle files')
        

else:
    # Load from cache (instant!)
    df = pd.read_pickle('df_cache.pkl')
    bl = pd.read_pickle('bl_cache.pkl')
    tminusone = pd.read_pickle('tminusone_cache.pkl')
    ste_df = pd.read_pickle('ste_df_cache.pkl')
    ste_bl = pd.read_pickle('ste_bl_cache.pkl')
    print('Loaded from cache')
    


Loaded from cache


In [14]:
df['application_received_dtm'] = pd.to_datetime(df['application_received_dtm'])

df['quarter'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('Q')
df['month'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('M')
df['week'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('W-SAT') #end on saturday



bl['time'] = pd.to_datetime(bl['time'])

bl['quarter'] = pd.to_datetime(bl['time']).dt.to_period('Q')
bl['month'] = pd.to_datetime(bl['time']).dt.to_period('M')
bl['week'] = pd.to_datetime(bl['time']).dt.to_period('W-SAT') #end on saturday








# df['con_ltv_back2'] = df['con_ltv_back'] *10.0652

count_acc_num = df['account_number'].nunique()

# countsds

#calculating
df['bbltv'] = df['con_amount_financed_back'] / df['bb_value'].replace(0, np.nan)





# df['atf_vantage'] = df['con_amount_financed_back']


# df['discount'] = np.where(df['lob'] == 'ENT', df['ent_disc'], 0)


df['luxury_flag'] = np.where( df['con_amount_financed_back'] >= 75000 , 1, 0)


# df['income_cb'] = df['income_cb'].fillna(0)

df['total_income'] = df['income_cb'].fillna(0) + df['income_pb'].fillna(0)


#assuming annual inflation of 3.5%, the average since 2020
# df['inflation_income'] = df['total_income'] / (1.035**(2025 - df['application_received_dtm'].dt.year.fillna(2025)  ))

# Set reference date as the start of 2025
# ref_date =

# Calculate number of weeks between application date and reference date
weeks_diff = ((pd.Timestamp.today() - df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
#
#
df['inflation_income'] = df['total_income'] / ( (1.035 ** (1/52)) ** weeks_diff)
#








# df['vantage'] = pd.to_numeric(df['vantage'], errors='coerce')
df['vantage2'] = np.where( (df['vantage'] >= 300) & (df['vantage'] <= 850)  , df['vantage'], np.nan    )




#df['ent_disc']




#filtering


#folters out the actual df similar to WHERE statement1
df = df[ df['application_received_dtm'] >= '2019-10-01']
# df = df[ df['application_received_dtm'] <= '2026-01-02']

# df = df[ df['con_amount_financed_back'] <= 75000]
df = df[ (df['lob'] == 'STE') | (df['con_amount_financed_back'] <= 75000)]
df = df[ df['con_pti_back'] <= 0.6]
# df = df[ df['bbltv'] <= 4.0]
# df = df[ (df['lob'] == 'MCY') | (df['lob'] == 'STE') | (df['bbltv'] <= 10.0) | (df['bb_value'].isna()) | (df['bb_value'] == 0) ]
df = df[ (df['lob'] == 'MCY') | (df['bbltv'] <= 10.0) | (df['bb_value'].isna()) | (df['bb_value'] == 0) ]
df = df[ df['total_income'] <= 200000]



# df = df[ df['bb_value'] > 1]
# df = df[ df['total_income'] <= 20000]




# df = df[(df['application_received_dtm'] > '2025-01-01')]





#if condition is TRUE then nan, else keep original value
# df['bbltv2'] = df['bbltv'].mask( (df['bbltv'] < 0.1) | (df['bbltv'] > 0.2) , np.nan)


#
# df['discount'] = np.select( [df['lob'] == 'ENT', df['data_source_id'] == 101]
#                             , [   df['ent_disc'] , df['aca_fee'] + df['processing_fee'] ]
#                             , np.nan)

df['discount'] = np.where( df['lob'] == 'ENT', df['ent_disc'] , df['disb_acquisition_fee_amt'] )


# --- STE cleaning ---
ste_df['application_received_dtm'] = pd.to_datetime(ste_df['application_received_dtm'])
ste_df['quarter'] = ste_df['application_received_dtm'].dt.to_period('Q')
ste_df['month'] = ste_df['application_received_dtm'].dt.to_period('M')
ste_df['week'] = ste_df['application_received_dtm'].dt.to_period('W-SAT')

ste_df['vantage2'] = ste_df['vantage']

ste_weeks_diff = ((pd.Timestamp.today() - ste_df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
ste_df['inflation_income'] = ste_df['total_income'] / ((1.035 ** (1/52)) ** ste_weeks_diff)

ste_df = ste_df[ ste_df['application_received_dtm'] >= '2019-10-01']
# ste_df = ste_df[ ste_df['con_amount_financed_back'] <= 75000]
ste_df = ste_df[ ste_df['con_pti_back'] <= 0.6]
ste_df = ste_df[ ste_df['total_income'] <= 200000]

df = pd.concat([df, ste_df], ignore_index=True)


#STE preintegration income is wrong in los deal current fact so we make it payment / PTI
ste_pre = (df['lob'] == 'STE') & (df['application_received_dtm'] < '2025-10-07')
df.loc[ste_pre, 'total_income'] = df.loc[ste_pre, 'con_payment_back_amt'] / df.loc[ste_pre, 'con_pti_back']
ste_pre_weeks = ((pd.Timestamp.today() - df.loc[ste_pre, 'application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
df.loc[ste_pre, 'inflation_income'] = df.loc[ste_pre, 'total_income'] / ((1.035 ** (1/52)) ** ste_pre_weeks)

ste_bl['time'] = pd.to_datetime(ste_bl['time'])
ste_bl['quarter'] = ste_bl['time'].dt.to_period('Q')
ste_bl['month'] = ste_bl['time'].dt.to_period('M')
ste_bl['week'] = ste_bl['time'].dt.to_period('W-SAT')

bl = pd.concat([bl, ste_bl], ignore_index=True)


VALID_LOBS = ['AN', 'ENT', 'FLD', 'FRN', 'STG', 'KMX', 'STE', 'MCY']

missing_lob_mask = ~df['lob'].isin(VALID_LOBS)
if missing_lob_mask.any():
    diag = df.loc[missing_lob_mask].groupby('week')['lob'].agg(
        missing_lob_contracts='count',
        lob_values=lambda x: x.unique().tolist()
    )
    print("--- DIAGNOSTIC: Contracts with unmapped LOBs ---")
    print(diag.to_string())
    print(f"Total excluded: {missing_lob_mask.sum()}")

df = df[df['lob'].isin(VALID_LOBS)].copy()
bl = bl[bl['lob'].isin(VALID_LOBS)].copy()

LOB_GENRE_MAP = {
    'KMX': 'KMX',
    'ENT': 'ENT',
    'STE': 'STE',
    'MCY': 'MCY',
    'AN': 'NonKMXENT',
    'STG': 'NonKMXENT',
    'FLD': 'NonKMXENT',
    'FRN': 'NonKMXENT',
}
df['lob_genre'] = df['lob'].map(LOB_GENRE_MAP)
bl['lob_genre'] = bl['lob'].map(LOB_GENRE_MAP)


mask = df['application_received_dtm'] >= '2023-01-01'
print(df.loc[mask, 'bbltv'].describe())

print(df['total_income'])



# df


--- DIAGNOSTIC: Contracts with unmapped LOBs ---
                       missing_lob_contracts      lob_values
week                                                        
2019-12-01/2019-12-07                      1              []
2019-12-08/2019-12-14                      2    [unassigned]
2019-12-15/2019-12-21                      1    [unassigned]
2019-12-22/2019-12-28                      8  [unassigned, ]
2019-12-29/2020-01-04                     16  [unassigned, ]
2020-01-05/2020-01-11                     12  [unassigned, ]
2020-01-12/2020-01-18                     17  [unassigned, ]
2020-01-19/2020-01-25                     20    [unassigned]
2020-01-26/2020-02-01                     18    [unassigned]
2020-02-02/2020-02-08                     14    [unassigned]
2020-02-09/2020-02-15                     15    [unassigned]
2020-02-16/2020-02-22                     12  [unassigned, ]
2020-02-23/2020-02-29                     44    [unassigned]
2020-03-01/2020-03-07               

In [15]:
#aggregating

granularity = ['quarter','month' , 'week'][0]
lob_granularity = ['lob_genre', 'lob'][0]



def generate_report(df, bl, granularity, lob_granularity):
    # Find two Saturdays ago
    today = pd.Timestamp.today().normalize()
    days_since_saturday = (today.weekday() - 5) % 7
    last_saturday = today - pd.Timedelta(days=days_since_saturday)
    two_saturdays_ago = last_saturday - pd.Timedelta(days=7)

    # Get periods
    week_ref = pd.Period(two_saturdays_ago, freq='W-SAT')
    month_ref = pd.Period(two_saturdays_ago, freq='M')

    if granularity == 'quarter':
        min_quarter = pd.Period('2020Q1')
        df = df[df['quarter'] >= min_quarter]
    if granularity == 'month':
        last_5_months = month_ref - 4
        df = df[(df['month'] >= last_5_months) & (df['month'] <= month_ref)]
    if granularity == 'week':
        if lob_granularity == 'lob':
            last_n_weeks = 12  # 13 weeks total (current + 12 previous)
        else:
            last_n_weeks = 5   # 6 weeks total (current + 5 previous)
        last_weeks = week_ref - last_n_weeks
        df  = df[(df['week'] >= last_weeks) & (df['week'] <= week_ref)]


        # last_6_weeks = week_ref - 5
        # df = df[(df['week'] >= last_6_weeks) & (df['week'] <= week_ref)]




    quarterly_df = df.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({



            # 'vantage' : (g['vantage2'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'vantage': (
                (g.loc[g['vantage2'].notnull(), 'vantage2'] * g.loc[g['vantage2'].notnull(), 'con_amount_financed_back']).sum()
                / g.loc[g['vantage2'].notnull(), 'con_amount_financed_back'].sum()
            ) if g.loc[g['vantage2'].notnull(), 'con_amount_financed_back'].sum() != 0 else np.nan,

            'model score weighted': (
                (g.loc[g['con_risk_model_score'].notnull(), 'con_risk_model_score'] * g.loc[g['con_risk_model_score'].notnull(), 'con_amount_financed_back']).sum()
                / g.loc[g['con_risk_model_score'].notnull(), 'con_amount_financed_back'].sum()
            ) if g.loc[g['con_risk_model_score'].notnull(), 'con_amount_financed_back'].sum() != 0 else np.nan,

            'cash down avg': g['con_cash_down_amt'].mean(),
            'cash down wtd': (
                (g['con_cash_down_amt'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum()
            ) if g['con_amount_financed_back'].sum() != 0 else np.nan,

            'discount avg': g['discount'].mean(),
            'discount pct': (
                g['discount'].sum() / g['con_amount_financed_back'].sum()
            ) if g['con_amount_financed_back'].sum() != 0 else np.nan,

            'amount financed avg': g['con_amount_financed_back'].mean(),

            'apr avg': g['con_apr'].mean(),
            'apr wtd': (
                (g['con_apr'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum()
            ) if g['con_amount_financed_back'].sum() != 0 else np.nan,

            'total income avg': g['total_income'].mean(),
            'inflation_adjusted_income avg': g['inflation_income'].mean(),

            'payment avg': g['con_payment_back_amt'].mean(),

            'pti wtd': (
                (g['con_pti_back'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum()
            ) if g['con_amount_financed_back'].sum() != 0 else np.nan,

            'contracts num': g['account_number'].nunique(),
            'duplicates': g['account_number'].duplicated().sum(),

        'all contracts': g['account_number'].count(),

            'Blackbook Value avg': g['bb_value'].mean(),
            'bbltv avg': g['bbltv'].mean(),
            'bbltv 2weighted': (
                (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum()
                / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum()
            ) if g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum() != 0 else np.nan,

            'mileage avg': g['purchase_odometer'].mean(),
            'vehicle age avg': g['veh_age'].mean(),


        })
    ).reset_index()




    b2l_df = bl.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({

            'b2l': g['cons'].sum() / g['apps'].sum() if g['apps'].sum() != 0 else np.nan

        })
    ).reset_index()


    merged_df = pd.merge(quarterly_df, b2l_df, on=[lob_granularity, granularity], how='left')



    melted = merged_df.melt(id_vars=[lob_granularity, granularity], var_name='measure', value_name='value')

    measure_order = [
        'vantage',
        'model score weighted', 'cash down avg',
        'cash down wtd', 'discount avg', 'discount pct', 'amount financed avg',
        'apr avg', 'apr wtd', 'total income avg',
        'inflation_adjusted_income avg',
        'payment avg', 'pti wtd', 'contracts num',
        'b2l',
        'Blackbook Value avg',
        'bbltv 2weighted', 'mileage avg',
        'vehicle age avg'
    ]

    melted = melted[melted['measure'].isin(measure_order)]

    pivoted = melted.pivot_table(index=[lob_granularity, 'measure'], columns=granularity, values='value', dropna=False)

    all_lobs = sorted(merged_df[lob_granularity].unique())
    full_index = pd.MultiIndex.from_product(
        [all_lobs, measure_order],
        names=[lob_granularity, 'measure']
    )
    pivoted = pivoted.reindex(full_index)

    return pivoted
    # pivoted















In [16]:


# generate_report(df, bl, 'quarter', 'lob_genre')
quarterly_df = generate_report(df, bl, 'quarter', 'lob_genre')

monthly_df = generate_report(df, bl, 'month', 'lob_genre')

weekly_df = generate_report(df, bl, 'week', 'lob_genre')

lob_breakout_df = generate_report(df, bl, 'week', 'lob')

quarterly_lob_breakout = generate_report(df, bl, 'quarter', 'lob')



quarterly_lob_breakout



quarter                                  2020Q1        2020Q2        2020Q3  \
lob measure                                                                   
AN  vantage                          530.608265    539.449229    545.305686   
    model score weighted             131.685561    133.946657    135.417557   
    cash down avg                   2519.534110   2868.730854   3488.931445   
    cash down wtd                   2631.192245   2960.561750   3930.683756   
    discount avg                    1563.964636   1759.772512   1730.676290   
    discount pct                       0.111201      0.132428      0.119824   
    amount financed avg            14064.333127  13288.530018  14443.501930   
    apr avg                            0.249378      0.252884      0.249411   
    apr wtd                            0.244210      0.249334      0.244298   
    total income avg                3578.106456   3586.493321   3703.372342   
    inflation_adjusted_income avg   2886.470754   2918.634815   3038.245773   
    payment avg                      386.803459    369.590875    396.410520   
    pti wtd                            0.142551      0.136145      0.143698   
    contracts num                   2229.000000   2216.000000   1751.000000   
    b2l                                0.108851      0.104307      0.110159   
    Blackbook Value avg             8370.028751   7881.664703  10133.057979   
    bbltv 2weighted                    1.899648      1.913873      1.629948   
    mileage avg                    92341.598475  90425.761733  85564.237579   
    vehicle age avg                    7.682107      7.548398      7.282125   
ENT vantage                          525.971326    533.006513    543.065541   
    model score weighted             132.732909    133.806178    136.147297   
    cash down avg                   2096.294414   2320.015900   2992.559651   
    cash down wtd                   1798.935361   2175.178073   2755.037011   
    discount avg                     700.706586   1044.081034    817.062838   
    discount pct                       0.041359      0.063442      0.045716   
    amount financed avg            16942.005577  16457.162561  17872.486717   
    apr avg                            0.224990      0.231263      0.231693   
    apr wtd                            0.224742      0.231790      0.232114   
    total income avg                3564.374370   3587.677282   3613.744761   
    inflation_adjusted_income avg   2876.784310   2917.979815   2965.100508   
...                                         ...           ...           ...   
STE apr wtd                                 NaN           NaN           NaN   
    total income avg                        NaN           NaN           NaN   
    inflation_adjusted_income avg           NaN           NaN           NaN   
    payment avg                             NaN           NaN           NaN   
    pti wtd                                 NaN           NaN           NaN   
    contracts num                           NaN           NaN           NaN   
    b2l                                     NaN           NaN           NaN   
    Blackbook Value avg                     NaN           NaN           NaN   
    bbltv 2weighted                         NaN           NaN           NaN   
    mileage avg                             NaN           NaN           NaN   
    vehicle age avg                         NaN           NaN           NaN   
STG vantage                          530.223711    539.642537    542.502607   
    model score weighted             130.092894    133.850336    135.564907   
    cash down avg                   2096.983543   2660.675002   3616.388873   
    cash down wtd                   2101.531905   2778.006363   4011.768999   
    discount avg                    1830.907832   1982.044208   1840.774926   
    discount pct                       0.131873      0.150729      0.129505   
    amount financed avg            13883.848108  1

In [17]:
# pivoted
#

In [18]:


# import pandas as pd

def format_time_columns(df):
    new_cols = []
    for col in df.columns:
        if isinstance(col, pd.Period):
            if col.freqstr == 'M':
                new_cols.append(col.start_time.strftime('%b %y'))
            elif col.freqstr.startswith('W'):
                new_cols.append(col.start_time.strftime('%d-%b-%y'))
            elif col.freqstr.startswith('Q'):
                new_cols.append(f"{col.year} Q{col.quarter}")
            else:
                new_cols.append(col)
        else:
            new_cols.append(col)
    df = df.copy()
    df.columns = new_cols
    return df

# Usage:
quarterly_fmt = format_time_columns(quarterly_df)
monthly_fmt = format_time_columns(monthly_df)
weekly_fmt = format_time_columns(weekly_df)
lob_breakout_fmt = format_time_columns(lob_breakout_df)
quarterly_lob_breakout_fmt = format_time_columns(quarterly_lob_breakout)













wb = openpyxl.load_workbook('pivoted_results.xlsx')
for sheet_name in ['lobgenre_quarter', 'lobgenre_month', 'lobgenre_week', 'lob_week', 'alllob_quarter']:
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for merge in list(ws.merged_cells.ranges):
            ws.merged_cells.remove(merge)
wb.save('pivoted_results.xlsx')
wb.close()

with pd.ExcelWriter('pivoted_results.xlsx', engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    quarterly_fmt.to_excel(writer, sheet_name='lobgenre_quarter', index=True)
    monthly_fmt.to_excel(writer, sheet_name='lobgenre_month', index=True)
    weekly_fmt.to_excel(writer, sheet_name='lobgenre_week', index=True)
    lob_breakout_fmt.to_excel(writer, sheet_name='lob_week', index=True)
    quarterly_lob_breakout_fmt.to_excel(writer, sheet_name='alllob_quarter', index=True)

